# Lane-drift monitor: run everything on Colab
CPU-only (classical CV), so the free runtime is enough. Steps: get the code, run tests, tune on validation clips, evaluate on held-out clips, build tables/figures, download the results.

Rough time on a free 2-vCPU runtime: tuning ~15 min, evaluation ~40 min with 12 clips per difficulty (use `CLIPS = 4` for a ~15 min smoke run).

In [ ]:
# 1) Get the code: either clone your GitHub repo, or upload lane-drift-monitor.zip
REPO_URL = ""   # e.g. "https://github.com/<user>/lane-drift-monitor.git"; leave empty to upload a zip
import os, subprocess
if REPO_URL:
    subprocess.run(["git", "clone", REPO_URL, "lane-drift-monitor"], check=True)
else:
    from google.colab import files
    up = files.upload()                      # choose lane-drift-monitor.zip
    name = next(iter(up))
    subprocess.run(["unzip", "-oq", name, "-d", "."], check=True)
%cd lane-drift-monitor
!pip -q install -r requirements.txt

In [ ]:
# 2) Unit tests (fast)
!python -m pytest -q tests

In [ ]:
# 3) Tune detector parameters on VALIDATION clips only (seeds 1000+). Writes configs/*.json
import os
WORKERS = os.cpu_count()
!sed -i "s/Pool(8/Pool({WORKERS}/" scripts/tune.py
!python scripts/tune.py --trials 60

In [ ]:
# 4) Held-out evaluation (seeds 0..CLIPS-1 per difficulty)
CLIPS = 12
!python scripts/run_eval.py --clips {CLIPS} --workers {WORKERS}

In [ ]:
# 5) Optional: real-photo sanity check. Upload the 25 course test images (test1.jpg ... test25.jpg) first.
import os
os.makedirs("data/real_stills", exist_ok=True)
from google.colab import files
up = files.upload()
for n, b in up.items():
    open(f"data/real_stills/{n}", "wb").write(b)
!python scripts/eval_real_stills.py

In [ ]:
# 6) Tables and figures
!python scripts/make_report.py
from IPython.display import Image, display
for f in ["ablation_bars", "trace_hard", "trace_medium", "lane_width_sensitivity"]:
    display(Image(f"outputs/figures/{f}.png"))

In [ ]:
# 7) Download results (summary.md, clip_scores.csv, figures, tuned configs)
!zip -qr results.zip outputs configs
from google.colab import files
files.download("results.zip")